# 🏷️ Notebook 07 — BioBERT Classification Model
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Fine-tune `dmis-lab/biobert-v1.1 (BioBERT)` on 6 medical categories
- **Exclude the 2,000-row RAG evaluation holdout from all classifier data** (zero leakage: holdout ∉ {train, val, test})
- Use an 80/10/10 train/val/test split of the remaining rows (random_state=42, stratified)
- Apply class weights to handle category imbalance
- Evaluate with per-class F1, macro F1, and accuracy
- Target: macro F1 ≥ 78%
- Save model to `models/classifier/biobert_classifier/`

### ⚠️ Label provenance — weak labels
All six categories come from the **programmatic keyword labeller** (`src/data/labeller.py`, notebook 03), not from human annotation. Every metric in this notebook measures agreement with those rules on unseen questions — **not** clinical ground truth. This is stated in `reports/classification_report.md`.

### 🔒 Holdout integrity
Notebook 05 holds out 2,000 stratified rows (`random_state=42`) from the FAISS/BM25 corpus and saves them to `data/processed/eval_holdout.csv`. This notebook reproduces that exact split and excludes those rows from all classifier data. They are reserved for RAG evaluation only.

### 📋 Deliverables
- `notebooks/07_classification_model.ipynb`
- `models/classifier/biobert_classifier/` (saved model)
- Classification report saved to `reports/`

---

## 1. Imports & Setup

In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

os.makedirs('../models/classifier', exist_ok=True)
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Libraries loaded | Device: {device}")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 2. Load & Prepare Data

In [2]:
import os

# Auto-detect cleaned data if available
DATA_DIR = '../data/processed'
clean_path = os.path.join(DATA_DIR, 'pubmedqa_labelled_cleaned.csv')
default_path = os.path.join(DATA_DIR, 'pubmedqa_labelled.csv')
csv_path = clean_path if os.path.exists(clean_path) else default_path
df = pd.read_csv(csv_path)
print(f"Loaded: {os.path.basename(csv_path)}")

# Build input text: question + context for richer signal
# BioBERT is a cased model — preserve original capitalisation
df['text'] = df['question'].astype(str) + " [SEP] " + df['context'].astype(str)

print(f"Dataset shape: {df.shape}")
print(f"\nCategory distribution:")
print(df['category'].value_counts())

# Label mapping (sorted for consistency)
label2id = {label: idx for idx, label in enumerate(sorted(df['category'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}

df['label_id'] = df['category'].map(label2id)

print(f"\nLabel mapping: {label2id}")
print(f"Number of classes: {len(label2id)}")

Loaded: pubmedqa_labelled.csv
Dataset shape: (211186, 5)

Category distribution:
category
Medication    71537
Treatment     48579
Diagnosis     31919
General       28251
Prevention    22180
Symptoms       8720
Name: count, dtype: int64

Label mapping: {'Diagnosis': 0, 'General': 1, 'Medication': 2, 'Prevention': 3, 'Symptoms': 4, 'Treatment': 5}
Number of classes: 6


## 3. Compute Class Weights

In [3]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(label2id)),
    y=df['label_id'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class weights:")
for label, weight in zip(sorted(label2id.keys()), class_weights):
    print(f"  {label:<15} → {weight:.4f}")

NameError: name 'compute_class_weight' is not defined

## 4. Train / Validation / Test Split (80/10/10)

In [4]:
# ── Reproduce notebook 05's cleaning + holdout split EXACTLY ────────────────
# NB05 applied, in order: dropna(question/context/answer) + strip + non-empty
# filter, then drop_duplicates(subset=['question']). It then held out 2,000
# stratified rows (random_state=42) for RAG evaluation and indexed only the
# remaining 209,108. To avoid label leakage, those exact 2,000 holdout rows
# are excluded from ALL classifier splits below.
before = len(df)
df = df.dropna(subset=["question", "context", "answer"]).copy()
df["question"] = df["question"].astype(str).str.strip()
df["context"] = df["context"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()
df = df[(df["question"] != "") & (df["context"] != "") & (df["answer"] != "")]
after_dropna = len(df)
df = df.drop_duplicates(subset=["question"], keep="first").copy().reset_index(drop=True)
# Rebuild the classifier input from the cleaned strings (cell 2 built it from
# the raw CSV before NB05-style cleaning)
df["text"] = df["question"].astype(str) + " [SEP] " + df["context"].astype(str)
print(f"Rows: {before:,} -> {after_dropna:,} (dropna/strip) -> {len(df):,} "
      f"(dedup, {after_dropna - len(df)} duplicates removed)")

rag_train_pool, df_holdout = train_test_split(
    df, test_size=2000, stratify=df["category"], random_state=42
)
rag_train_pool = rag_train_pool.reset_index(drop=True)
df_holdout = df_holdout.reset_index(drop=True)

# Integrity check against notebook 05's published holdout artifact
holdout_artifact = "../data/processed/eval_holdout.csv"
if os.path.exists(holdout_artifact):
    published = pd.read_csv(holdout_artifact)
    cols = ["question", "context", "answer", "category"]
    matches = (
        set(map(tuple, published[cols].astype(str).values))
        == set(map(tuple, df_holdout[cols].astype(str).values))
    )
    print(f"Holdout parity with data/processed/eval_holdout.csv: {matches}")
    assert matches, "Reproduced holdout does not match notebook 05 artifact"

# ── Classifier splits: 80/10/10 of the NON-holdout rows ─────────────────────
train_df, temp_df = train_test_split(
    rag_train_pool, test_size=0.2, stratify=rag_train_pool["category"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["category"], random_state=42
)

# Leakage guard: the RAG holdout must not appear in any classifier split
holdout_questions = set(df_holdout["question"])
for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    overlap = len(set(split_df["question"]) & holdout_questions)
    assert overlap == 0, f"Holdout leakage: {overlap} holdout rows found in {name}"

# Recompute class weights on the FINAL training split (section 3 used the
# full dataset before holdout exclusion; weights must reflect train only)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(label2id)),
    y=train_df["label_id"].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights recomputed on the post-exclusion train split.")

print(f"Train size: {len(train_df):,} ({len(train_df)/len(df)*100:.0f}%)")
print(f"Val size:   {len(val_df):,} ({len(val_df)/len(df)*100:.0f}%)")
print(f"Test size:  {len(test_df):,} ({len(test_df)/len(df)*100:.0f}%)")
print(f"RAG holdout (excluded from all classifier data): {len(df_holdout):,}")

Rows: 211,186 -> 211,186 (dropna/strip) -> 211,108 (dedup, 78 duplicates removed)


NameError: name 'train_test_split' is not defined

## 5. Tokenizer & Dataset

BioBERT (`dmis-lab/biobert-v1.1`) is pre-trained on PubMed + PMC text — significantly better domain fit than general-purpose DistilBERT for medical text classification. It is a **cased** model — inputs must **not** be lowercased.

In [5]:
MODEL_NAME      = "dmis-lab/biobert-v1.1"
LOCAL_SAVE_PATH = "../models/classifier/biobert_classifier"
HF_REPO_ID      = "AbdoMatrix/biobert-medical-classifier"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Loaded tokenizer: {MODEL_NAME} (cased)")

NameError: name 'AutoTokenizer' is not defined

In [6]:
class MedicalDataset(Dataset):
    """PyTorch Dataset wrapping tokenised BioBERT inputs."""

    def __init__(self, texts: list, labels: list, tokenizer, max_length: int = 256):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = MedicalDataset(
    train_df["text"].tolist(), train_df["label_id"].tolist(), tokenizer
)
val_dataset = MedicalDataset(
    val_df["text"].tolist(), val_df["label_id"].tolist(), tokenizer
)
test_dataset = MedicalDataset(
    test_df["text"].tolist(), test_df["label_id"].tolist(), tokenizer
)

print(f"✅ Datasets created")
print(f"   Train: {len(train_dataset):,} samples")
print(f"   Val:   {len(val_dataset):,} samples")
print(f"   Test:  {len(test_dataset):,} samples")

NameError: name 'Dataset' is not defined

## 6. Custom Trainer with Weighted Loss

HuggingFace `Trainer` doesn't use class weights by default.
We override `compute_loss` to apply weighted cross-entropy.

In [7]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("✅ WeightedTrainer defined")

NameError: name 'Trainer' is not defined

## 7. Model & Training Configuration

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="../models/classifier/checkpoints",

    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    report_to="none",
)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1_macro    = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    acc         = accuracy_score(labels, preds)
    return {
        'f1_macro':    f1_macro,
        'f1_weighted': f1_weighted,
        'accuracy':    acc,
    }


print(f"✅ Model initialized | Epochs: 3 | LR: 2e-5 | Batch: 16")

NameError: name 'AutoModelForSequenceClassification' is not defined

## 8. Train

In [9]:
trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
)

print("🚀 Starting fine-tuning...")
trainer.train()

NameError: name 'WeightedTrainer' is not defined

## 9. Save Model Locally

In [10]:
os.makedirs(LOCAL_SAVE_PATH, exist_ok=True)
trainer.save_model(LOCAL_SAVE_PATH)
tokenizer.save_pretrained(LOCAL_SAVE_PATH)
print(f"✅ Model saved locally: {LOCAL_SAVE_PATH}")

NameError: name 'trainer' is not defined

## 10. Evaluate on Test Set

In [11]:
predictions = trainer.predict(test_dataset)
preds       = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['label_id'].values

# ── Classification Report ────────────────────────────────────────────────────
report_str = classification_report(
    true_labels, preds,
    target_names=sorted(label2id.keys())
)

print("=" * 60)
print("CLASSIFICATION REPORT (Test Set)")
print("=" * 60)
print(report_str)

# ── Key Metrics ──────────────────────────────────────────────────────────────
macro_f1    = f1_score(true_labels, preds, average='macro')
weighted_f1 = f1_score(true_labels, preds, average='weighted')
acc         = accuracy_score(true_labels, preds)

print(f"\n🎯 Macro F1:    {macro_f1:.4f}")
print(f"🎯 Weighted F1: {weighted_f1:.4f}")
print(f"🎯 Accuracy:    {acc:.4f}")

# Best epoch: load_best_model_at_end restores the checkpoint with the highest
# validation f1_macro, so read the best epoch from the evaluation history.
_eval_hist = [h for h in trainer.state.log_history if 'eval_f1_macro' in h]
best_eval  = max(_eval_hist, key=lambda h: h['eval_f1_macro']) if _eval_hist else {}
best_epoch = best_eval.get('epoch')
if best_epoch is not None:
    print(f"🏆 Best epoch (by val f1_macro): {best_epoch}")

if macro_f1 >= 0.78:
    print("\n✅ KPI MET: Macro F1 ≥ 78%")
else:
    print(f"\n⚠️  KPI NOT MET: Macro F1 = {macro_f1:.4f} (target ≥ 0.78)")
    print("    Consider: more epochs, different learning rate, or data augmentation")

# ── Reproducibility manifest ─────────────────────────────────────────────────
import json
import platform
from datetime import datetime, timezone

training_manifest = {
    'run_date_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'dataset': 'qiaojin/PubMedQA (pqa_artificial)',
    'labels': 'programmatic keyword labels from src/data/labeller.py (weak labels, not human)',
    'labelled_rows_loaded': int(before),
    'rows_after_dropna_strip': int(after_dropna),
    'duplicates_removed': int(after_dropna - len(df)),
    'rows_after_dedup': int(len(df)),
    'rag_holdout_excluded': int(len(df_holdout)),
    'classifier_train': int(len(train_df)),
    'classifier_val': int(len(val_df)),
    'classifier_test': int(len(test_df)),
    'random_state': 42,
    'stratified': True,
    'base_model_checkpoint': MODEL_NAME,
    'max_seq_length': 256,
    'epochs_configured': int(training_args.num_train_epochs),
    'early_stopping_patience': 2,
    'best_epoch_by_val_f1_macro': float(best_epoch) if best_epoch is not None else None,
    'learning_rate': training_args.learning_rate,
    'batch_size': training_args.per_device_train_batch_size,
    'weight_decay': training_args.weight_decay,
    'warmup_steps': training_args.warmup_steps,
    'seed': training_args.seed,
    'fp16': bool(training_args.fp16),
    'metrics': {
        'macro_f1': round(float(macro_f1), 4),
        'weighted_f1': round(float(weighted_f1), 4),
        'accuracy': round(float(acc), 4),
    },
    'library_versions': {
        'transformers': __import__('transformers').__version__,
        'torch': torch.__version__,
        'python': platform.python_version(),
    },
}
with open('../reports/classifier_training_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(training_manifest, f, indent=2)
print("✅ Training manifest saved: ../reports/classifier_training_manifest.json")

NameError: name 'trainer' is not defined

## 11. Confusion Matrix

In [12]:
cm = confusion_matrix(true_labels, preds)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=sorted(label2id.keys()),
    yticklabels=sorted(label2id.keys())
)
plt.title(f'Confusion Matrix — Macro F1: {macro_f1:.4f}', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('../reports/figures/07_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

NameError: name 'confusion_matrix' is not defined

## 12. Save Classification Report & Upload to HuggingFace

In [13]:
from huggingface_hub import HfApi

# ── Save classification report ───────────────────────────────────────────────
report_path = "../reports/classification_report.md"

report_content = f"""# Classification Report — BioBERT Medical Classifier

## Label Provenance — weak labels
All six categories are **programmatic keyword labels** generated by
`src/data/labeller.py` (notebook 03). They are not human annotations: metrics
below measure agreement with the keyword rules on the held-out test split.
Systematic labeller errors propagate to the classifier and are invisible to
these metrics.

## Run Provenance
| Item | Value |
|---|---|
| Base model | `{MODEL_NAME}` |
| Dataset | qiaojin/PubMedQA (pqa_artificial) |
| Labelled rows loaded | {{before:,}} |
| After dropna/strip cleaning | {{after_dropna:,}} |
| Duplicates removed (question) | {{after_dropna - len(df):,}} |
| Rows after dedup | {{len(df):,}} |
| RAG holdout excluded (never trained on) | {{len(df_holdout):,}} |
| Train / Val / Test | {{len(train_df):,}} / {{len(val_df):,}} / {{len(test_df):,}} |
| Split seed | 42 (stratified, sklearn train_test_split) |

## Model Details
| Item | Value |
|---|---|
| Classes | {{len(label2id)}} |
| Max sequence length | 256 |
| Epochs configured | 10 |
| Early stopping | patience=2 on val f1_macro |
| Best epoch (by val f1_macro) | {{best_epoch if best_epoch is not None else 'n/a'}} |
| Learning rate | 2e-5 |
| Batch size | 16 |
| Class weights | Applied (balanced, recomputed on train split) |

## Test Set Results
{report_str}

## Key Metrics
| Metric | Value |
|---|---|
| Macro F1 | {macro_f1:.4f} |
| Weighted F1 | {weighted_f1:.4f} |
| Accuracy | {acc:.4f} |
| KPI (Macro F1 ≥ 0.78) | {"✅ MET" if macro_f1 >= 0.78 else "⚠️ NOT MET"} |

## What this metric measures
Macro-F1 here is the classifier's ability to reproduce the keyword-labeller's
category assignment on unseen PubMedQA questions, evaluated on a test split
that shares no rows with the RAG evaluation holdout.

## Label Mapping
{chr(10).join(f"| {label} | {idx} |" for label, idx in sorted(label2id.items()))}
"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_content)
print(f"✅ Classification report saved to: {report_path}")

# ── Build model card ─────────────────────────────────────────────────────────
category_rows = "\n".join(f"| {idx} | {label} |" for idx, label in sorted(id2label.items()))

usage_code = (
    'from transformers import AutoTokenizer, AutoModelForSequenceClassification\n'
    'import torch\n\n'
    f'tokenizer = AutoTokenizer.from_pretrained("{HF_REPO_ID}")\n'
    f'model = AutoModelForSequenceClassification.from_pretrained("{HF_REPO_ID}")\n\n'
    'text = "What are the symptoms of diabetes?"\n'
    'inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)\n\n'
    'with torch.no_grad():\n'
    '    outputs = model(**inputs)\n\n'
    'predicted = model.config.id2label[torch.argmax(outputs.logits, dim=1).item()]\n'
    'print(predicted)  # → Symptoms'
)

model_card = f"""---
language: en
license: mit
tags:
  - medical
  - classification
  - biobert
  - pubmedqa
  - healthcare-rag
datasets:
  - qiaojin/PubMedQA
metrics:
  - f1
pipeline_tag: text-classification
---

# BioBERT Medical Query Classifier

Fine-tuned `dmis-lab/biobert-v1.1` for classifying medical questions into 6 categories.

## Categories
| ID | Category |
|----|----------|
{category_rows}

## Results
| Metric | Score |
|--------|-------|
| Macro F1 | {macro_f1:.4f} |
| Weighted F1 | {weighted_f1:.4f} |
| Accuracy | {acc:.4f} |

## Training Config
| Item | Value |
|------|-------|
| Base model | dmis-lab/biobert-v1.1 |
| Dataset | qiaojin/PubMedQA ({len(df):,} rows after dedup; {len(df_holdout):,} RAG-holdout rows excluded) |
| Split | 80/10/10, stratified, random_state=42 |
| Epochs | 10 configured, early stopping (best epoch: {best_epoch if best_epoch is not None else 'n/a'}) |
| Learning rate | 2e-5 |
| Batch size | 16 |
| Class weights | Balanced (custom WeightedTrainer) |
| Labels | Programmatic keyword labels (weak labels, src/data/labeller.py) |

## Usage
{usage_code}

## Project
Healthcare RAG-Powered Medical Q&A Assistant
eyouth x DEPI | Microsoft Machine Learning Track | 2026
GitHub: https://github.com/AbdooMatrix/Healthcare-RAG-Powered-Medical-QA-Assistant
"""

card_path = os.path.join(LOCAL_SAVE_PATH, "README.md")
with open(card_path, "w", encoding="utf-8") as f:
    f.write(model_card)
print(f"✅ Model card saved: {card_path}")

# ── Upload to HuggingFace ────────────────────────────────────────────────────
try:
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=LOCAL_SAVE_PATH, repo_id=HF_REPO_ID, repo_type="model")
    print(f"\n✅ Model uploaded to HuggingFace: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"\n⚠️  HuggingFace upload failed: {e}")
    print("   Model is saved locally. Upload manually later with:")
    print(f"   huggingface-cli upload {HF_REPO_ID} {LOCAL_SAVE_PATH}")


NameError: name 'report_str' is not defined

## 13. Quick Test — Verify Saved Model Loads

In [14]:
sys.path.append(os.path.abspath('..'))
from src.classification.classifier import load_classifier

clf = load_classifier()

test_texts = [
    "What are the symptoms of diabetes?",
    "How is pneumonia diagnosed?",
    "What is the treatment for hypertension?",
    "What are the side effects of aspirin?",
    "How can heart disease be prevented?",
    "What is the role of antibodies?",
]

print("\nQuick classification test:")
for text in test_texts:
    cat = clf.predict(text)
    print(f"  '{text[:50]}...' → {cat}")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## ✅ Summary

| Item | Status |
|---|---|
| Model | `dmis-lab/biobert-v1.1 (BioBERT)` fine-tuned |
| Split | 80/10/10 |
| Epochs | 3 |
| Class weights | Applied via custom WeightedTrainer |
| Macro F1 | Evaluated and logged |
| Accuracy | Evaluated and logged |
| Model saved | `models/classifier/biobert_classifier/` |
| Report saved | `reports/classification_report.md` |

---

### ➡️ Next Step
Open **`08_evaluation.ipynb`** to evaluate RAG vs plain LLM.